# 01 · Exploratory Data Analysis
## Credit Card Fraud Detection

**Dataset:** Kaggle ULB Credit Card Fraud (2013)  
**Features:** Time, V1–V28 (PCA-transformed), Amount  
**Target:** Class (0 = Legitimate, 1 = Fraud)

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Notebook-wide style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor': '#1a1a2e',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#334155',
    'grid.color': '#334155',
    'figure.dpi': 120,
})

DATA_PATH = '../creditcard.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Basic Info & Missing Values

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

## 2. Class Distribution (Imbalance)

In [ ]:
counts = df['Class'].value_counts()
fraud_rate = counts[1] / len(df) * 100

print(f'Legitimate transactions : {counts[0]:,} ({100 - fraud_rate:.2f}%)')
print(f'Fraudulent transactions : {counts[1]:,} ({fraud_rate:.4f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
bars = axes[0].bar(['Legitimate', 'Fraud'], counts.values,
                   color=['#6366f1', '#ef4444'], edgecolor='none', width=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold', color='#e2e8f0')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', va='bottom', color='#e2e8f0', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'],
            colors=['#6366f1', '#ef4444'],
            autopct='%1.3f%%', startangle=90,
            textprops={'color': '#e2e8f0'},
            wedgeprops={'edgecolor': '#0f0f1a', 'linewidth': 2})
axes[1].set_title('Fraud vs Legitimate (%)', fontsize=14, fontweight='bold', color='#e2e8f0')

plt.tight_layout()
plt.savefig('../data/processed/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, color, cls in zip(axes, ['Legitimate', 'Fraud'], ['#6366f1', '#ef4444'], [0, 1]):
    data = df[df['Class'] == cls]['Amount']
    ax.hist(data, bins=50, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(f'{label} — Amount Distribution', fontsize=13, color='#e2e8f0', fontweight='bold')
    ax.set_xlabel('Amount (USD)')
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='white', linestyle='--', linewidth=1.5,
               label=f'Median: ${data.median():.2f}')
    ax.legend()

plt.tight_layout()
plt.show()

print('=== Amount Statistics by Class ===')
df.groupby('Class')['Amount'].describe()

## 4. Transaction Time Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(df[df['Class'] == 0]['Time'] / 3600, bins=100, alpha=0.7,
        color='#6366f1', label='Legitimate', edgecolor='none')
ax.hist(df[df['Class'] == 1]['Time'] / 3600, bins=100, alpha=0.9,
        color='#ef4444', label='Fraud', edgecolor='none')
ax.set_xlabel('Hours since first transaction')
ax.set_ylabel('Count')
ax.set_title('Transaction Time Distribution — Fraud vs Legitimate',
             fontsize=13, color='#e2e8f0', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Correlation Heatmap (Top Features vs Class)

In [ ]:
# Correlations with Class
corr_with_class = df.corr()['Class'].drop('Class').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top positive and negative correlations
top_pos = corr_with_class.tail(10)
top_neg = corr_with_class.head(10)

for ax, data, title, color in zip(
    axes,
    [top_neg, top_pos],
    ['Most Negatively Correlated with Fraud', 'Most Positively Correlated with Fraud'],
    ['#6366f1', '#ef4444'],
):
    bars = ax.barh(data.index, data.values, color=color, alpha=0.85, edgecolor='none')
    ax.set_title(title, fontsize=12, color='#e2e8f0', fontweight='bold')
    ax.set_xlabel('Pearson Correlation with Class')
    ax.axvline(0, color='white', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.savefig('../data/processed/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Full Correlation Heatmap (V1–V28 + Amount)

In [ ]:
cols_to_plot = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Time', 'Class']
corr_matrix = df[cols_to_plot].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, ax=ax,
    cmap='coolwarm', center=0,
    linewidths=0.3, linecolor='#0f0f1a',
    cbar_kws={'shrink': 0.8},
    annot=False,
)
ax.set_title('Feature Correlation Heatmap', fontsize=15, color='#e2e8f0', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Box Plots — Key Features vs Class

In [ ]:
# Features most correlated with fraud
key_features = ['V14', 'V12', 'V10', 'V4', 'V11', 'V2', 'Amount']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    data_legit = df[df['Class'] == 0][feat]
    data_fraud = df[df['Class'] == 1][feat]
    ax.boxplot(
        [data_legit, data_fraud],
        labels=['Legit', 'Fraud'],
        patch_artist=True,
        boxprops=dict(facecolor='#1e1e3a'),
        medianprops=dict(color='#a78bfa', linewidth=2),
        whiskerprops=dict(color='#94a3b8'),
        capprops=dict(color='#94a3b8'),
        flierprops=dict(marker='o', markerfacecolor='#ef4444', markersize=2, alpha=0.3),
    )
    ax.set_title(feat, fontsize=12, color='#e2e8f0', fontweight='bold')
    ax.set_ylabel('Value')

axes[-1].set_visible(False)  # hide last empty plot
plt.suptitle('Distribution of Key Features — Legitimate vs Fraud',
             fontsize=14, color='#e2e8f0', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Summary

Key EDA findings:

1. **Extreme imbalance** — only 0.172% of transactions are fraud. Standard accuracy is useless; we'll use AUPRC.
2. **V14, V12, V10** show the strongest negative correlation with fraud — these will be the most important features.
3. **V4, V11** show the strongest positive correlation with fraud.
4. **Fraud amounts** tend to be lower on average than legitimate amounts — fraudsters tend to avoid large flagged amounts.
5. **Time** — fraud occurs across both day cycles; no clear temporal clustering.
6. V1–V28 are already PCA-normalised; only `Time` and `Amount` need scaling.

→ **Next:** `02_modelling.ipynb` — training, SMOTE, evaluation